In [1]:
from utils import *
from FAPT import *
from DRAG import *
 
simulation = Simulation()
dim_q1, dim_q2, dim_c = (3,3,3)

# Steuerungs-Flag für symbolische oder numerische Belegung
symbolic = True
tg_val = 250.0  # Gate-Zeit [ns]
sigma_r_g_val = 0.3
sigma_r_t_val = 0.1

In [2]:
import time
import numpy as np
import sympy as sp
from IPython.display import Math, display


print(
    f"[Block 1] Initialisiere Parameter, Energien & hermitesches V1"
    f" (symbolic={symbolic})..."
)
start_time = time.perf_counter()

# 1.1 Zeit- und Puls-Symbole
t_sym = sp.Symbol("t", real=True, positive=True)
tg_sym = sp.Symbol("t_g", real=True, positive=True)
sigma_sym = sp.Symbol(r"\sigma", real=True, positive=True)

# 1.2 Frequenz- und Steuer-Symbole
wd0_sym = sp.Symbol(r"\omega_{d0}", real=True)
dwd_t_sym = sp.Function(r"\delta\omega_d", real=True)(t_sym)
wd_t_sym = wd0_sym + dwd_t_sym

# Komplexes A(t) = A_r(t) + i A_i(t)
Ar_sym = sp.Function("A_r", real=True)(t_sym)
Ai_sym = sp.Function("A_i", real=True)(t_sym)
A_t_sym = Ar_sym + sp.I * Ai_sym

# Freie physikalische System-Parameter
Delta_sym = sp.Symbol("Delta", real=True)
lam_sym = sp.Symbol("lambda", real=True)

# 2. Dynamic State/Matrix Init (Symbolisch vs. Numerisch)
D = 27
V0_sym = sp.zeros(D, D)

if symbolic:
  # --- SYMBOLISCHER PFAD ---
  E_array_sym = np.array([sp.Symbol(f"E_{i}", real=True) for i in range(D)])

  V1_dressed_ref = simulation.V1_dressed_array
  threshold = 1e-8
  V1_sym = np.zeros((D, D), dtype=object)

  for i1 in range(dim_q1):
    for i2 in range(dim_q2):
      for ic in range(dim_c):
        for jc in range(dim_c):
          for j1 in range(dim_q1):
            for j2 in range(dim_q2):
              i = i1 * dim_c * dim_q2 + ic * dim_q2 + i2
              j = j1 * dim_c * dim_q2 + jc * dim_q2 + j2

              if np.abs(V1_dressed_ref[i, j]) >= threshold:
                real = i1 == j1 and ic == jc and i2 == j2
                bra_ket = sp.Symbol(
                    f"\\bra{{{i1}{ic}{i2}}}\\widetilde{{V_1}}{{\\ket{{{j1}{jc}{j2}}}}}",
                    real=real,
                )
                V1_sym[i, j] = bra_ket
              else:
                V1_sym[i, j] = sp.S.Zero
                  # Physikalische Parameter
else:
  # --- NUMERISCHER PFAD ---
  E_array_sym = np.array(simulation.E_array, dtype=object)
  V1_sym = np.array(simulation.V1_dressed_array, dtype=object)


[Block 1] Initialisiere Parameter, Energien & hermitesches V1 (symbolic=True)...


In [ ]:
print("[Block 2] Konstruiere gefilterte symbolische Matrix V1 (V0 = 0) und starte FAPT...")
start_time = time.perf_counter()


# 2.3 FAPT Heff symbolisch aufrufen
print("  ➜ Berechne Heff via FAPT analytisch...", end="", flush=True)
heff_start = time.perf_counter()

Heff_sym = Heff_Floquet_total_matrix_summed(
    rH=getattr(simulation, 'rH', 1),
    wd=wd_t_sym,
    A=A_t_sym,
    resonances=simulation.resonances,
    E=E_array_sym,                  # REIN SYMBOLISCHE ENERGIEN
    V_posHarm=V1_sym,                # GEFILTERTE SYMBOLISCHE BRA-KET MATRIX
    V0=V0_sym,                       # EXPLIZIT SP.ZEROS
    ref_state=getattr(simulation, 'ref_state', None),
    dwd=0,
    dA=0,
    t=t_sym,
    analytics=True,
    include_geometric=False,
    include_micromotion=False,
    include_g_correction=False,
    verbose=False,
    rW=getattr(simulation, 'rW', 1)
)

print(f" Fertig in {time.perf_counter() - heff_start:.2f}s!")

# ==============================================================================
# SYMBOLISCHE KONDENSATION / LINEARISIERUNG
# ==============================================================================
def condense_element_symbolic(expr, var, order=1, factor_vars=None):
    if expr == 0 or expr == sp.S.Zero:
        return sp.S.Zero
    if factor_vars is None:
        factor_vars = []

    terms = sp.Add.make_args(expr)
    linearized_terms = []

    for term in terms:
        if not term.has(var):
            linearized_terms.append(sp.cancel(term))
            continue

        f_0 = term.subs(var, 0)
        if order == 0:
            linearized_terms.append(sp.cancel(f_0))
        elif order == 1:
            df_0 = term.diff(var).subs(var, 0)
            linearized_terms.append(sp.cancel(f_0) + sp.cancel(df_0) * var)

    total_sum = sp.Add(*linearized_terms)
    poly_dwd = sp.Poly(total_sum, var)
    coeffs = poly_dwd.all_coeffs()
    degree = poly_dwd.degree()

    compact_terms = []
    for idx, coeff in enumerate(coeffs):
        current_pow = degree - idx
        coeff_factored = sp.factor(sp.collect(coeff, factor_vars))
        if current_pow == 0:
            compact_terms.append(coeff_factored)
        else:
            compact_terms.append(coeff_factored * (var**current_pow))

    return sp.Add(*compact_terms)

# Linearisierung der symbolischen Matrix
H_eff_raw = sp.Matrix(Heff_sym)
delta_shift = H_eff_raw[0, 0]
H_eff_shifted = H_eff_raw - delta_shift * sp.eye(H_eff_raw.shape[0])

rows, cols = H_eff_shifted.rows, H_eff_shifted.cols
Heff = sp.zeros(rows, cols)

for i in range(rows):
    for j in range(cols):
        elem = H_eff_shifted[i, j]
        if elem != 0:
            Heff[i, j] = condense_element_symbolic(elem, dwd_t_sym, order=1, factor_vars=[Ar_sym, Ai_sym])

if symbolic:
    Delta_val = -1.2
    lambda_val = 10.2
else:
    Delta_sol = -2*Heff[1,1] + Heff[2,2]
    lambda_val = Heff[1,2] / Heff[0,1]

print(f"✅ Block 2 abgeschlossen in {time.perf_counter() - start_time:.2f}s\n")
display(Math(rf"(H_{{\text{{eff}}}})_{{0,1}} = {sp.latex(Heff[0, 1])}"))
display(Math(rf"(H_{{\text{{eff}}}})_{{1,2}} = {sp.latex(Heff[1,2])}"))

[Block 2] Konstruiere gefilterte symbolische Matrix V1 (V0 = 0) und starte FAPT...
  ➜ Berechne Heff via FAPT analytisch... Fertig in 0.02s!
✅ Block 2 abgeschlossen in 0.05s



<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [4]:
print("[Block 3] Berechne H_target via DRAG mit symbolischen Zeitparametern...")
start_time = time.perf_counter()

H_target_raw, subs_ht = h_target_symbolic_abstract(
    Delta=Delta_sym,
    rrr=lam_sym, 
    tg=tg_sym,               # SYMBOLISCHES tg
    sigma_r=sigma_sym, # SYMBOLISCHES sigma (sigma_r = sigma / tg)
    pulse='gauss', 
    t=t_sym, 
    order=4
)

H_target = sp.Matrix(H_target_raw).doit()

print(f"✅ Block 3 abgeschlossen in {time.perf_counter() - start_time:.2f}s\n")

[Block 3] Berechne H_target via DRAG mit symbolischen Zeitparametern...
✅ Block 3 abgeschlossen in 0.24s



In [5]:
import time
import sympy as sp
from IPython.display import display, Math

print("[Block 4] Extrahiere die 3 Bestimmungsgleichungen aus H_eff und H_target...")
start_time = time.perf_counter()

# 1. Differenzmatrix bilden
H_diff = Heff - H_target

# 3. Dynamische Extraktion aus den Matrixelementen und Ersetzung anwenden
raw_re_01 = sp.re(H_diff[0, 1])
raw_im_01 = sp.im(H_diff[0, 1])
raw_diag_11 = sp.re(H_diff[1, 1])

# Gleichungen mit explizitem "= 0"
equations = [
    sp.Eq(raw_re_01, 0),
    sp.Eq(raw_im_01, 0),
    sp.Eq(raw_diag_11, 0)
]

print(f"✅ Block 4 abgeschlossen in {time.perf_counter() - start_time:.2f}s")
print("  -> 3 Gleichungen dynamisch extrahiert.\n")

# 4. Strings für LaTeX-Ausgabe aufbereiten
str_eq1 = sp.latex(equations[0])
str_eq2 = sp.latex(equations[1])
str_eq3 = sp.latex(equations[2])

display(Math(rf"\text{{1. }} A_r(t)\text{{-Gl. (Re 0,1)}}: \quad {str_eq1}"))
display(Math(rf"\text{{2. }} A_i(t)\text{{-Gl. (Im 0,1)}}: \quad {str_eq2}"))
display(Math(rf"\text{{3. }} \delta\omega_d(t)\text{{-Gl. (Diag 1,1)}}: \quad {str_eq3}"))

# 5. LaTeX-Code inklusive Definition der Hilfsfunktionen generieren
latex_system_code = rf"""% --- Definition der Hilfsfunktionen ---

% --- Bestimmungsgleichungen ---
\begin{{aligned}}
  \text{{1. }} A_r(t)\text{{-Gl. (Re 0,1)}} &: \quad {str_eq1} \\[2ex]
  \text{{2. }} A_i(t)\text{{-Gl. (Im 0,1)}} &: \quad {str_eq2} \\[2ex]
  \text{{3. }} \delta\omega_d(t)\text{{-Gl. (Diag 1,1)}} &: \quad {str_eq3}
\end{{aligned}}"""

# 6. Ausgabe als Roh-Text zum Kopieren
print("\n--- REINER LATEX-CODE (ZUM KOPIEREN) ---")
print(latex_system_code)

[Block 4] Extrahiere die 3 Bestimmungsgleichungen aus H_eff und H_target...
✅ Block 4 abgeschlossen in 0.01s
  -> 3 Gleichungen dynamisch extrahiert.



<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>


--- REINER LATEX-CODE (ZUM KOPIEREN) ---
% --- Definition der Hilfsfunktionen ---

% --- Bestimmungsgleichungen ---
\begin{aligned}
  \text{1. } A_r(t)\text{-Gl. (Re 0,1)} &: \quad - \frac{A_{i}{\left(t \right)} \operatorname{im}{\left(\bra{100}\widetilde{V_1}{\ket{001}}\right)}}{2} + \frac{A_{r}{\left(t \right)} \operatorname{re}{\left(\bra{100}\widetilde{V_1}{\ket{001}}\right)}}{2} - \frac{\operatorname{{\mathcal E}}_\pi{\left(t \right)}}{2} = 0 \\[2ex]
  \text{2. } A_i(t)\text{-Gl. (Im 0,1)} &: \quad - \frac{A_{i}{\left(t \right)} \operatorname{re}{\left(\bra{100}\widetilde{V_1}{\ket{001}}\right)}}{2} - \frac{A_{r}{\left(t \right)} \operatorname{im}{\left(\bra{100}\widetilde{V_1}{\ket{001}}\right)}}{2} - \frac{\dot{\mathcal E}_\pi}{2 \Delta} = 0 \\[2ex]
  \text{3. } \delta\omega_d(t)\text{-Gl. (Diag 1,1)} &: \quad - E_{1} + E_{9} - \omega_{d0} - \delta\omega_{d}{\left(t \right)} - \frac{\left(\lambda^{2} - 4\right) \operatorname{{\mathcal E}}_\pi^{2}{\left(t \right)}}{4 \Delta} = 0

In [6]:
# 1. dwd aus Gl. 3 auflösen
sol_dwd = sp.solve(equations[2], dwd_t_sym)[0]

# 2. Gleichungen vorerst normal expandieren
V1_diag_list = list(V1_sym.diagonal())

eq1_expanded = equations[0].expand(complex=True)
eq2_expanded = equations[1].expand(complex=True)

# 3. A_i und A_r bestimmen
sol_Ai_from_eq2 = sp.solve(eq2_expanded, Ai_sym)[0]
sol_Ar_from_eq1 = sp.solve(eq1_expanded, Ar_sym)[0]

eq_Ar = sp.Eq(sol_Ar_from_eq1.subs(Ai_sym, sol_Ai_from_eq2), Ar_sym)
sol_Ar = sp.solve(eq_Ar, Ar_sym)[0]
sol_Ai = sol_Ai_from_eq2.subs(Ar_sym, sol_Ar)

# 5. LaTeX-Strings erstellen
str_Ar = sp.latex(sp.Eq(Ar_sym, sol_Ar))
str_Ai = sp.latex(sp.Eq(Ai_sym, sol_Ai))
str_dwd = sp.latex(sp.Eq(dwd_t_sym, sol_dwd))

# 6. Anzeige aller drei Lösungen
display(Math(rf"\text{{Hauptamplitude }} A_r(t): \quad {str_Ar}"))
display(Math(rf"\text{{DRAG-Korrektur }} A_i(t): \quad {str_Ai}"))
display(Math(rf"\text{{Frequenzkorrektur }} \delta\omega_d(t): \quad {str_dwd}"))
# 7. Reiner LaTeX-Code zum Kopieren
latex_output = rf"""\begin{{aligned}}
  A_r(t) &= {sp.latex(sol_Ar)} \\[2ex]
  A_i(t) &= {sp.latex(sol_Ai)} \\[2ex]
  \delta\omega_d(t) &= {sp.latex(sol_dwd)}
\end{{aligned}}"""

print("\n--- REINER LATEX-CODE (ZUM KOPIEREN) ---")
print(latex_output)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>


--- REINER LATEX-CODE (ZUM KOPIEREN) ---
\begin{aligned}
  A_r(t) &= \frac{\Delta \operatorname{{\mathcal E}}_\pi{\left(t \right)} \operatorname{re}{\left(\bra{100}\widetilde{V_1}{\ket{001}}\right)} - \dot{\mathcal E}_\pi \operatorname{im}{\left(\bra{100}\widetilde{V_1}{\ket{001}}\right)}}{\Delta \left(\left(\operatorname{re}{\left(\bra{100}\widetilde{V_1}{\ket{001}}\right)}\right)^{2} + \left(\operatorname{im}{\left(\bra{100}\widetilde{V_1}{\ket{001}}\right)}\right)^{2}\right)} \\[2ex]
  A_i(t) &= \frac{- \dot{\mathcal E}_\pi - \frac{\left(\Delta \operatorname{{\mathcal E}}_\pi{\left(t \right)} \operatorname{re}{\left(\bra{100}\widetilde{V_1}{\ket{001}}\right)} - \dot{\mathcal E}_\pi \operatorname{im}{\left(\bra{100}\widetilde{V_1}{\ket{001}}\right)}\right) \operatorname{im}{\left(\bra{100}\widetilde{V_1}{\ket{001}}\right)}}{\left(\operatorname{re}{\left(\bra{100}\widetilde{V_1}{\ket{001}}\right)}\right)^{2} + \left(\operatorname{im}{\left(\bra{100}\widetilde{V_1}{\ket{001}}\right)}\

In [ ]:
# Determination of wd0 
ep1, ep2 = subs_ht.keys()
wd0_eq = sp.Eq(sol_dwd.subs({t_sym:0, ep1:0, ep2:0}),0)
wd0_sol = sp.solve(wd0_eq, wd0_sym)[0]
if not symbolic:
    Delta_val = Delta_sol.subs({wd0_sym: wd0_sol})
Delta_val

-1.2

In [8]:
import time
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp

# Paper-Style Plotting Layout
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 18,
    'axes.labelsize': 18,
    'axes.titlesize': 18,
    'legend.fontsize': 14,
    'xtick.labelsize': 15,
    'ytick.labelsize': 15,
    'lines.linewidth': 2.5,
    'figure.dpi': 600,
    'mathtext.fontset': 'cm',
})

fig, axes = plt.subplots(2, 2, figsize=(15, 11))

# ==============================================================================
# 1. MODULARISIERTE FUNKTION FÜR PULS-BERECHNUNG
# ==============================================================================


def process_pulse_data(
    pulse_type,
    sigma_val,
    tg_val,
    lambda_val,
    sol_Ar,
    sol_Ai,
    sol_dwd,
    Delta_sym,
    lam_sym,
    tg_sym,
    sigma_sym,
    t_sym,
):
  print(
      f"--- Berechne H_target für '{pulse_type}' (sigma_r = {sigma_val}) ---"
  )
  start_time = time.perf_counter()

  # 1. H_target symbolisch berechnen
  _, subs_ht = h_target_symbolic_abstract(
      Delta=Delta_sym,
      rrr=lam_sym,
      tg=tg_sym,
      sigma_r=sigma_sym,
      pulse=pulse_type,
      t=t_sym,
      order=4,
  )

  # 2. Exakte symbolische Bestimmung von wd0 über t=0 & ep1=0, ep2=0
  ep1, ep2 = list(subs_ht.keys())[:2]
  wd0_symbols = [
      s
      for s in sol_dwd.free_symbols
      if any(k in s.name.lower() for k in ['omega', 'wd0', 'd0'])
  ]

  sol_Ar_corr, sol_Ai_corr, sol_dwd_corr = sol_Ar, sol_Ai, sol_dwd
  if wd0_symbols:
    wd0_eq = sp.Eq(sol_dwd.subs({t_sym: 0, ep1: 0, ep2: 0}), 0)
    wd0_sol = sp.solve(wd0_eq, wd0_symbols[0])
    if wd0_sol:
      sol_Ar_corr = sol_Ar.subs(wd0_symbols[0], wd0_sol[0])
      sol_Ai_corr = sol_Ai.subs(wd0_symbols[0], wd0_sol[0])
      sol_dwd_corr = sol_dwd.subs(wd0_symbols[0], wd0_sol[0])

  # 3. Symbole zusammenführen & Parameter-Map aufbauen
  ht_exprs = [v for v in subs_ht.values() if isinstance(v, sp.Basic)]
  combined_symbols = (
      sol_Ar_corr.free_symbols
      | sol_Ai_corr.free_symbols
      | sol_dwd_corr.free_symbols
      | set().union(*(e.free_symbols for e in ht_exprs))
  )

  params = {}
  for s in combined_symbols:
    nl = s.name.lower()
    if 'sigma' in nl or s == sigma_sym:
      params[s] = sigma_val
    elif 't_g' in nl or 'tg' in nl or s == tg_sym:
      params[s] = tg_val
    elif 'Delta' in nl or s == Delta_sym:
      params[s] = Delta_val
    elif 'lambda' in nl or s == lam_sym:
      params[s] = lambda_val
    elif s != t_sym:
      if '100' in s.name or '001' in s.name:
        params[s] = 1.0 + 0.01j
      elif 'E' in s.name and '_' in s.name and s.name.split('_')[1].isdigit():
        params[s] = simulation.E_array[int(s.name.split('_')[1])]
      else:
        params[s] = 1.0

  # 4. Pulsfunktionen auflösen
  numeric_func_map = {
      k: (v.doit().subs(params) if isinstance(v, sp.Basic) else v)
      for k, v in subs_ht.items()
  }

  Ar_expr_sub = sol_Ar_corr.subs(numeric_func_map).subs(params).doit()
  Ai_expr_sub = sol_Ai_corr.subs(numeric_func_map).subs(params).doit()
  dwd_expr_sub = sol_dwd_corr.subs(numeric_func_map).subs(params).doit()

  # Lambdifizieren für Plot
  Ar_func = sp.lambdify(t_sym, Ar_expr_sub, modules=['numpy'])
  Ai_func = sp.lambdify(t_sym, Ai_expr_sub, modules=['numpy'])
  dwd_func = sp.lambdify(t_sym, dwd_expr_sub, modules=['numpy'])

  # 5. Numerische Auswertung
  def safe_eval(func, t_arr):
    res = []
    for t_v in t_arr:
      try:
        v = func(t_v)
        res.append(0.0 if np.isnan(v) or np.isinf(v) else v)
      except Exception:
        res.append(0.0)
    return np.array(res)

  t_vals = np.linspace(0.0, tg_val, 1000)
  Ar_vals = np.real_if_close(safe_eval(Ar_func, t_vals)).real
  Ai_vals = np.real_if_close(safe_eval(Ai_func, t_vals)).real
  dwd_vals = np.real_if_close(safe_eval(dwd_func, t_vals)).real

  if dwd_vals.shape != t_vals.shape:
    dwd_vals = np.full_like(t_vals, fill_value=float(dwd_vals.squeeze()))

  print(f'   Fertig in {time.perf_counter() - start_time:.2f}s\n')

  return (
      t_vals / tg_val,
      Ar_vals,
      Ai_vals,
      dwd_vals,
      Ar_expr_sub,
      Ai_expr_sub,
      dwd_expr_sub,
  )


# ==============================================================================
# 2. AUSFÜHRUNG & PLOTTING
# ==============================================================================

pulses = [
    (
        'gauss',
        sigma_r_g_val,
        'Gaussian DRAG Envelope',
        'Gaussian Dynamic Shift',
        axes[0],
    ),
    ('tanh', sigma_r_t_val, 'Tanh DRAG Envelope', 'Tanh Dynamic Shift', axes[1]),
]


def setup_subplot(ax, label, title=''):
  ax.grid(True, linestyle=':', alpha=0.7)
  ax.axhline(0, color='black', linewidth=1.0, alpha=0.5)
  ax.set_xlim(0, 1)
  ax.set_xlabel(r'Time $t/t_g$')
  if title:
    ax.set_title(title, pad=12)
  ax.text(
      0.03,
      0.88,
      f'({label})',
      transform=ax.transAxes,
      fontsize=22,
      fontweight='bold',
      va='top',
  )


labels = [['a', 'b'], ['c', 'd']]

for idx, (p_type, sig_val, title1, title2, (ax1, ax2)) in enumerate(pulses):
  (
      t_norm,
      Ar_raw,
      Ai_raw,
      dwd_raw,
      Ar_expr_sub,
      Ai_expr_sub,
      dwd_expr_sub,
  ) = process_pulse_data(
      p_type,
      sig_val,
      tg_val,
      lambda_val,
      sol_Ar,
      sol_Ai,
      sol_dwd,
      Delta_sym,
      lam_sym,
      tg_sym,
      sigma_sym,
      t_sym,
  )

  max_Ar = np.max(np.abs(Ar_raw)) or 1.0
  max_dwd = np.max(np.abs(dwd_raw)) or 1.0

  ax1.plot(t_norm, Ar_raw / max_Ar, color='#1f4e78', label=r'$A_r(t) / \max|A_r|$')
  ax1.plot(
      t_norm,
      10 * (Ai_raw / max_Ar),
      color='#a61c1c',
      label=r'$10\times A_i(t) / \max|A_r|$',
  )
  ax1.set_ylabel('Normalized Envelopes')
  ax1.legend(loc='upper right', frameon=True, edgecolor='black')
  setup_subplot(ax1, labels[idx][0], title=rf'{title1} ($\sigma_r = {sig_val}$)')

  ax2.plot(
      t_norm,
      dwd_raw / max_dwd,
      color='#127352',
      label=rf'$\delta\omega_d(t) / \max|\delta\omega_d|$',
  )
  ax2.set_ylabel('Normalized Frequency Deviation')
  ax2.legend(loc='upper right', frameon=True, edgecolor='black')
  setup_subplot(ax2, labels[idx][1], title=rf'{title2} ($\sigma_r = {sig_val}$)')

plt.tight_layout()
plt.savefig('Figure/drag_pulse_shapes.pdf', bbox_inches='tight')
plt.show()

--- Berechne H_target für 'gauss' (sigma_r = 0.3) ---
   Fertig in 0.18s

--- Berechne H_target für 'tanh' (sigma_r = 0.1) ---
   Fertig in 0.03s



# Now save the data

In [9]:
from pathlib import Path
import numpy as np
import sympy as sp

# ==============================================================================
# SPEICHER-FUNKTION (Kompakt)
# ==============================================================================


def save_analytical_drive_data(
    tg: float,
    sigma_ratio: float,
    pulse_type: str,
    sol_Ar: sp.Expr,
    sol_Ai: sp.Expr,
    sol_dwd: sp.Expr,
    wd0_val: float = 0.0,
    fit_dir: str | Path = 'operational_res/fit_data',
):
  fit_path = Path(fit_dir)
  fit_path.mkdir(parents=True, exist_ok=True)
  file_path = (
      fit_path
      / f'fit_params_{pulse_type}_tg_{round(tg)}ns_sigma_{sigma_ratio:.1f}.npz'
  )

  np.savez_compressed(
      file_path,
      A_r_expr=str(sp.cancel(sol_Ar)),
      A_i_expr=str(sp.cancel(sol_Ai)),
      dwd_expr=str(sp.cancel(sol_dwd)),
      wd0=float(wd0_val),
  )
  print(f"💾 Daten erfolgreich gespeichert unter: '{file_path}'")


# ==============================================================================
# ANPASSUNG IN PROCESS_PULSE_DATA (Rückgabe der SymPy-Ausdrücke)
# ==============================================================================
# Ändere das return am Ende von process_pulse_data zu:
# return t_vals / tg_val, Ar_vals, Ai_vals, dwd_vals, Ar_sub_final, Ai_sub_final, dwd_sub_final


# ==============================================================================
# COMPACT EXECUTION & SAVE LOOP
# ==============================================================================
if not symbolic:
  save_configs = [
      ('gauss', sigma_r_g_val),
      ('tanh', sigma_r_t_val),
  ]

  for p_type, sig_val in save_configs:
    t_norm, Ar_raw, Ai_raw, dwd_raw, Ar_expr, Ai_expr, dwd_expr = (
        process_pulse_data(
            p_type,
            sig_val,
            tg_val,
            lambda_val,
            sol_Ar,
            sol_Ai,
            sol_dwd,
            Delta_sym,
            lam_sym,
            tg_sym,
            sigma_sym,
            t_sym,
        )
    )

    save_analytical_drive_data(
        tg=tg_val,
        sigma_ratio=sig_val,
        pulse_type=p_type,
        sol_Ar=Ar_expr,
        sol_Ai=Ai_expr,
        sol_dwd=dwd_expr,
        wd0_val=wd0_sol,
    )